In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import os
import warnings
import matplotlib.patches as mpatches

%matplotlib inline
warnings.filterwarnings('ignore')

In [ ]:
# Define file paths for data and output figures
data_path = os.path.dirname(os.getcwd()) + '/data'
figure_path = os.path.dirname(os.getcwd()) + '/figures'

# Selected proteins to plot

In [ ]:
# Load BCP-ALL subtype specific biomarkers
df = pd.read_excel(data_path + '/results/results.xlsx', 
                   sheet_name='Table S15 HeH vs ER-overlap', 
                   skiprows=1)

# Create lists for each subtype
ER_top_proteins = df[df['Up regulated group'] == 'ETV6::RUNX1']['Protein /gene'].tolist()
HeH_top_proteins = df[df['Up regulated group'] == 'HeH']['Protein /gene'].tolist()

# Display results
print(f"ETV6::RUNX1 proteins: {len(ER_top_proteins)}")
print(ER_top_proteins)
print(f"\nHeH proteins: {len(HeH_top_proteins)}")
print(HeH_top_proteins)

# Olink data and clinical information

In [ ]:
# Load and preprocess Olink proteomics data
olink = pd.read_csv(data_path + '/raw/olink.csv', delimiter=';')
olink = olink[['SampleID', 'Assay', 'NPX']]  # Keep only essential columns
print('Olink data - number of unique Sample IDs', len(set(olink['SampleID'])))

# Pivot data from long to wide format (samples as rows, proteins as columns)
pivoted_data = olink.pivot_table(index='SampleID', columns='Assay', values='NPX', aggfunc='mean')
pivoted_data.columns.name = None
pivoted_data = pivoted_data.reset_index()

# Load sample phenotype information
pheno = pd.read_excel(data_path + '/raw/MASTER_PHENO_FILE_AML_ALL_controls_20260122.xlsx')

# Clean immunopheno column - replace missing values with 'CTRL' (control)
pheno['immunopheno'] = pheno['immunopheno'].fillna('Control').replace(['nan', 'NaN', None], 'Control')
pheno['SampleID'] = pheno['sample_id']
print('Pheno data - number of unique Sample IDs', len(set(pheno['SampleID'])))
pheno = pheno[['SampleID', 'public_id', 'immunopheno', 'Subtype']]

# Merge proteomics data with phenotype information
merged = pd.merge(pivoted_data, pheno, on='SampleID', how='inner')

# Remove specific patients (quality control exclusions)
removed_patients = ['AML_101','AML_139', 'ALL_920', 'K-023']
final_df = merged[~merged['public_id'].isin(removed_patients)]
print('Merged data - number of unique Sample IDs', len(set(final_df['SampleID'])))

print('----------')
print('Sample distribution across immunophenotypes')
counts = final_df['immunopheno'].value_counts()
print(counts)

In [ ]:
colors = {
    'BCP-ALL': '#447597',
    'AML': '#EFB2D1',
    'Control': '#A6A6A6',
    'T-ALL': '#87CCB2'
}

category_order = ['Control', 'AML', 'T-ALL', 'BCP-ALL\nER', 'BCP-ALL\nHeH']

In [ ]:
def _prep_long(final_df: pd.DataFrame, proteins: list[str]) -> pd.DataFrame:
    df = final_df.copy()

    if 'Subtype' in df.columns:
        df['Subtype'] = df['Subtype'].replace({'t1221': 'ETV6::RUNX1'})

    # --- MODIFY immunopheno column ---
    if 'immunopheno' in df.columns:
        is_ball = df['immunopheno'].eq('B-ALL')

        # map subtypes to the new immunopheno labels (two-line labels using \n)
        subtype_to_immuno = {
            'ETV6::RUNX1': 'BCP-ALL\nER',
            'HeH': 'BCP-ALL\nHeH',
        }

        df.loc[is_ball, 'immunopheno'] = df.loc[is_ball, 'Subtype'].map(subtype_to_immuno)

    # after the rewrite, immunopheno is the single plotting column
    df['plot_group'] = df.get('immunopheno', np.nan)

    # long format
    keep = ['SampleID', 'plot_group'] + [p for p in proteins if p in df.columns]
    missing = [p for p in proteins if p not in df.columns]
    if missing:
        print(f"[WARN] Missing proteins not in final_df columns: {missing}")

    dfl = df[keep].melt(
        id_vars=['SampleID', 'plot_group'],
        value_vars=[p for p in proteins if p in df.columns],
        var_name='Protein',
        value_name='NPX'
    ).dropna(subset=['plot_group', 'NPX'])

    # enforce order
    dfl['plot_group'] = pd.Categorical(dfl['plot_group'], categories=category_order, ordered=True)

    return dfl

def plot_biomarker_panel(final_df: pd.DataFrame, proteins: list[str], panel_title: str,
                         ncols: int = 5, figsize_per_ax=(4, 3.8)):
    dfl = _prep_long(final_df, proteins)

    n = len(proteins)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))

    fig_w = figsize_per_ax[0] * ncols
    fig_h = figsize_per_ax[1] * nrows + 0.8
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), squeeze=False)

    # colors (BCP-ALL split shares same color)
    box_face = {
        'Control': colors['Control'],
        'AML': colors['AML'],
        'T-ALL': colors['T-ALL'],
        'BCP-ALL\nER': colors['BCP-ALL'],
        'BCP-ALL\nHeH': colors['BCP-ALL'],
    }

    for i, prot in enumerate(proteins):
        ax = axes[i // ncols][i % ncols]

        sub = dfl[dfl['Protein'] == prot]
        if sub.empty:
            ax.text(0.5, 0.5, f'{prot}\nNot found', ha='center', va='center', transform=ax.transAxes)
            ax.set_axis_off()
            continue

        data, labels = [], []
        for g in category_order:
            vals = sub.loc[sub['plot_group'] == g, 'NPX'].astype(float).values
            data.append(vals)
            labels.append(g)

        bp = ax.boxplot(
            data,
            positions=np.arange(1, len(category_order) + 1),
            widths=0.62,
            patch_artist=True,
            showfliers=False,
            medianprops=dict(color='black', linewidth=1.4),
            whiskerprops=dict(color='#474646', linewidth=1.2),
            capprops=dict(color='#474646', linewidth=1.2),
        )

        for j, box in enumerate(bp['boxes']):
            g = labels[j]
            box.set_facecolor(box_face.get(g, '#CCCCCC'))
            box.set_edgecolor('#474646')
            box.set_linewidth(1.2)
            box.set_alpha(0.55)

            # hatch one of the BCP-ALL subtypes (optional)
            if g == 'BCP-ALL\nER':
                box.set_hatch('///')
                box.set_alpha(0.45)

        rng = np.random.default_rng(7)
        for j, g in enumerate(labels, start=1):
            vals = sub.loc[sub['plot_group'] == g, 'NPX'].astype(float).values
            if len(vals) == 0:
                continue

            x = j + rng.normal(0, 0.06, size=len(vals))
            marker = '^' if g == 'BCP-ALL\nER' else 'o'

            ax.scatter(
                x, vals,
                s=28,
                marker=marker,
                facecolor=box_face.get(g, '#CCCCCC'),
                edgecolor='grey',
                linewidth=0.8,
                alpha=0.9,
                zorder=3
            )

        ax.set_title(prot, fontsize=11, fontweight='bold')
        ax.set_xticks(np.arange(1, len(category_order) + 1))
        ax.set_xticklabels(labels, rotation=0, ha='center', fontsize=9)  # two-line labels read best w/ 0 rotation
        ax.set_ylabel('NPX', fontsize=10)
        ax.grid(True, axis='y', alpha=0.25)
        ax.tick_params(axis='y', labelsize=9)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    for k in range(n, nrows * ncols):
        axes[k // ncols][k % ncols].set_axis_off()

    fig.suptitle(panel_title, fontsize=14, fontweight='bold', y=0.995)

    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D

    legend_items = [
        Patch(facecolor=colors['Control'], edgecolor='#474646', alpha=0.55, label='Control'),
        Patch(facecolor=colors['AML'], edgecolor='#474646', alpha=0.55, label='AML'),
        Patch(facecolor=colors['T-ALL'], edgecolor='#474646', alpha=0.55, label='T-ALL'),
        Patch(facecolor=colors['BCP-ALL'], edgecolor='#474646', alpha=0.55, label='BCP-ALL'),
        Line2D([0], [0], marker='o', color='none', markerfacecolor='#CCCCCC', markeredgecolor='grey',
               markersize=7, label='BCP-ALL (HeH)'),
        Line2D([0], [0], marker='^', color='none', markerfacecolor='#CCCCCC', markeredgecolor='grey',
               markersize=7, label='BCP-ALL (ER)'),
    ]
    fig.legend(handles=legend_items, loc='lower center', ncol=3, frameon=True,
               bbox_to_anchor=(0.5, -0.01), fontsize=10)

    fig.tight_layout(rect=[0, 0.06, 1, 0.96])
    return fig

In [ ]:
fig1 = plot_biomarker_panel(
    final_df,
    ER_top_proteins,
    panel_title='',
    ncols=5
)
plt.show()

In [ ]:
category_order = [
    'Control',
    'AML',
    'T-ALL',
    'BCP-ALL\nHeH',
    'BCP-ALL\nER',
]

fig2 = plot_biomarker_panel(
    final_df,
    HeH_top_proteins,
    panel_title='',
    ncols=5
)
plt.show()

# fig1.savefig("updated_figures/panel_ETV6_RUNX1_boxplots.png", dpi=300, bbox_inches="tight")
# fig2.savefig("updated_figures/panel_HeH_boxplots.png", dpi=300, bbox_inches="tight")

# Main plots

In [ ]:
def p_to_stars(p):
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def compute_per_protein_stats(dfl: pd.DataFrame, subtype_label: str,
                              refs=('Control', 'AML', 'T-ALL'),
                              method='fdr_bh') -> pd.DataFrame:
    out_rows = []

    # 1) Compute and store ALL raw p-values (one row per protein x comparison)
    for prot, sub in dfl.groupby('Protein'):
        a = sub.loc[sub['plot_group'] == subtype_label, 'NPX'].astype(float).dropna().values

        for ref in refs:
            b = sub.loc[sub['plot_group'] == ref, 'NPX'].astype(float).dropna().values

            # Handle edge cases
            if len(a) == 0 or len(b) == 0:
                p = np.nan
            else:
                p = mannwhitneyu(a, b, alternative='two-sided').pvalue

            out_rows.append({
                "Protein": prot,
                "comparison": f"{subtype_label} vs {ref}",
                "p": float(p) if np.isfinite(p) else np.nan,
            })

    res = pd.DataFrame(out_rows)

    # 2) Global BH-FDR across ALL valid tests
    pvals = res["p"].to_numpy(dtype=float)
    ok = np.isfinite(pvals)

    res["p_adj"] = np.nan
    if ok.sum() > 0:
        res.loc[ok, "p_adj"] = multipletests(pvals[ok], method=method)[1]

    # 3) Stars based on adjusted p-values
    res["stars"] = res["p_adj"].apply(lambda pa: p_to_stars(pa) if np.isfinite(pa) else "NA")

    return res

In [ ]:
def annotate_stats_text(ax, stats_df: pd.DataFrame, prot: str, subtype_short: str):
    s = stats_df[stats_df["Protein"] == prot].copy()
    if s.empty:
        return

    # create compact lines (you can rename labels however you want)
    lines = []
    for _, row in s.iterrows():
        # row['comparison'] looks like "BCP-ALL\nETV6::RUNX1 vs Control"
        if row["comparison"].endswith("vs Control"):
            lab = "vs Ctrl"
        elif row["comparison"].endswith("vs AML"):
            lab = "vs AML"
        elif row["comparison"].endswith("vs T-ALL"):
            lab = "vs T-ALL"
        else:
            lab = row["comparison"].split(" vs ")[-1]

        lines.append(f"{lab}: {row['stars']}")

    txt = "\n".join(lines)

    ax.text(
        0.05, 0.98,
        txt,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=8,
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="#cccccc", alpha=0.85),
        zorder=10
    )

In [ ]:
def plot_biomarker_panel(final_df: pd.DataFrame, proteins: list[str], panel_title: str,
                         ncols: int = 5, figsize_per_ax=(4, 3.8),
                         subtype_label: str | None = None,
                         add_stats: bool = True,
                         p_adjust_method: str = "fdr_bh"):

    dfl = _prep_long(final_df, proteins)

    stats_df = None
    if add_stats:
        if subtype_label is None:
            raise ValueError("If add_stats=True, provide subtype_label (e.g. 'BCP-ALL\\nETV6::RUNX1').")
        stats_df = compute_per_protein_stats(dfl, subtype_label=subtype_label, method=p_adjust_method)

    n = len(proteins)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))

    fig_w = figsize_per_ax[0] * ncols
    fig_h = figsize_per_ax[1] * nrows + 0.8
    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), squeeze=False)

    box_face = {
        'Control': colors['Control'],
        'AML': colors['AML'],
        'T-ALL': colors['T-ALL'],
        'BCP-ALL\nER': colors['BCP-ALL'],
        'BCP-ALL\nHeH': colors['BCP-ALL'],
    }

    for i, prot in enumerate(proteins):
        ax = axes[i // ncols][i % ncols]
        sub = dfl[dfl['Protein'] == prot]

        if sub.empty:
            ax.text(0.5, 0.5, f'{prot}\nNot found', ha='center', va='center', transform=ax.transAxes)
            ax.set_axis_off()
            continue

        data, labels = [], []
        for g in category_order:
            vals = sub.loc[sub['plot_group'] == g, 'NPX'].astype(float).values
            data.append(vals)
            labels.append(g)

        bp = ax.boxplot(
            data,
            positions=np.arange(1, len(category_order) + 1),
            widths=0.62,
            patch_artist=True,
            showfliers=False,
            medianprops=dict(color='black', linewidth=1.4),
            whiskerprops=dict(color='#474646', linewidth=1.2),
            capprops=dict(color='#474646', linewidth=1.2),
        )

        for j, box in enumerate(bp['boxes']):
            g = labels[j]
            box.set_facecolor(box_face.get(g, '#CCCCCC'))
            box.set_edgecolor('#474646')
            box.set_linewidth(1.2)
            box.set_alpha(0.55)
            if g == 'BCP-ALL\nER':
                box.set_hatch('///')
                box.set_alpha(0.45)

        rng = np.random.default_rng(7)
        for j, g in enumerate(labels, start=1):
            vals = sub.loc[sub['plot_group'] == g, 'NPX'].astype(float).values
            if len(vals) == 0:
                continue
            x = j + rng.normal(0, 0.06, size=len(vals))
            marker = '^' if g == 'BCP-ALL\nER' else 'o'
            ax.scatter(
                x, vals,
                s=28,
                marker=marker,
                facecolor=box_face.get(g, '#CCCCCC'),
                edgecolor='grey',
                linewidth=0.8,
                alpha=0.9,
                zorder=3
            )

        ax.set_title(prot, fontsize=11, fontweight='bold')
        ax.set_xticks(np.arange(1, len(category_order) + 1))
        ax.set_xticklabels(labels, rotation=0, ha='center', fontsize=9)
        ax.set_ylabel('NPX', fontsize=10)
        ax.grid(True, axis='y', alpha=0.25)
        ax.tick_params(axis='y', labelsize=9)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        # --- add stats text ---
        if add_stats and stats_df is not None:
            annotate_stats_text(ax, stats_df, prot=prot, subtype_short="")

    for k in range(n, nrows * ncols):
        axes[k // ncols][k % ncols].set_axis_off()

    fig.suptitle(panel_title, fontsize=14, fontweight='bold', y=0.995)

    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    legend_items = [
        Patch(facecolor=colors['Control'], edgecolor='#474646', alpha=0.55, label='Control'),
        Patch(facecolor=colors['AML'], edgecolor='#474646', alpha=0.55, label='AML'),
        Patch(facecolor=colors['T-ALL'], edgecolor='#474646', alpha=0.55, label='T-ALL'),
        Patch(facecolor=colors['BCP-ALL'], edgecolor='#474646', alpha=0.55, label='BCP-ALL'),
        Line2D([0], [0], marker='o', color='none', markerfacecolor='#CCCCCC', markeredgecolor='grey',
               markersize=7, label='BCP-ALL (HeH)'),
        Line2D([0], [0], marker='^', color='none', markerfacecolor='#CCCCCC', markeredgecolor='grey',
               markersize=7, label='BCP-ALL (ER)'),
    ]
    fig.legend(handles=legend_items, loc='lower center', ncol=3, frameon=True,
               bbox_to_anchor=(0.5, -0.01), fontsize=10)

    fig.tight_layout(rect=[0, 0.06, 1, 0.96])
    return fig

In [ ]:
category_order = ['Control', 'AML', 'T-ALL', 'BCP-ALL\nER', 'BCP-ALL\nHeH']
fig1 = plot_biomarker_panel(
    final_df,
    ER_top_proteins,
    panel_title='',
    ncols=5,
    subtype_label='BCP-ALL\nER',
    add_stats=True
)
plt.show()

#category_order = ['Control', 'AML', 'T-ALL', 'BCP-ALL\nHeH', 'BCP-ALL\nER']
fig2 = plot_biomarker_panel(
    final_df,
    HeH_top_proteins,
    panel_title='',
    ncols=5,
    subtype_label='BCP-ALL\nHeH',
    add_stats=True
)
plt.show()


In [ ]:
fig1.savefig("updated_figures/panel_ETV6_RUNX1_boxplots.pdf", dpi=300, bbox_inches="tight")
fig2.savefig("updated_figures/panel_HeH_boxplots.pdf", dpi=300, bbox_inches="tight")
fig1.savefig("updated_figures/panel_ETV6_RUNX1_boxplots.png", dpi=300, bbox_inches="tight")
fig2.savefig("updated_figures/panel_HeH_boxplots.png", dpi=300, bbox_inches="tight")